# Vector RAG Pipeline (Tuned) on the OrchestrAI Synthetic Enterprise Corpus

This notebook is a tuned variant of `rag_vectordb.ipynb` with larger chunk sizes to evaluate the impact of chunking parameters on retrieval quality.

Changes compared with the original `rag_vectordb` notebook:
- `HybridChunker` `max_tokens` increased from **300** to **400**
- `DocumentSplitter` `split_length` increased from **80** to **200** words
- `DocumentSplitter` `split_overlap` increased from **20** to **40** words
- results are saved under `implementation="vectordb_tuned"`

Everything else (questions, prompt, LLM, embedding model, `top_k=10`) is identical to isolate the effect of chunking.

**Outputs and IDs:** Logged as `implementation="vectordb_tuned"` with scenario IDs **`vt1`–`vt5`** (same five questions as `rag_vectordb`) to `data/results/experimental/hpc_results.csv`. Each run clears only previous `vectordb_tuned` rows before saving. See `README.md` for evaluation.

## What is different from the basic notebook?

### Basic pipeline
- direct DOCX conversion
- simple splitting
- in-memory document store
- simpler indexing flow

### Advanced pipeline
- richer conversion and chunking
- persistent vector storage in Milvus
- more robust source metadata handling
- a more realistic enterprise-style retrieval pipeline

## Environment requirements

This notebook assumes:
- Python **3.12** (recommended) and dependencies from `requirements.txt`
- vLLM services are running and reachable from this HPC job/session
- `/nvme/scratch/cy26nk2/llm.env` and `/nvme/scratch/cy26nk2/embd.env` exist with valid API keys, URLs, and model names
- Milvus is available in this environment (the notebook uses a file-backed URI under `notebooks-hpc/`)
- the OrchestrAI corpus under **`data/dummy_data/`** (`PROJECT_ROOT / "data" / "dummy_data"`)
- the first code cell prepends **`src/`** to `sys.path` for `results_logger`

Run the optional install cell once if packages are missing.

## Imports and runtime checks

In [1]:
import sys, os
import warnings
import time
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "notebooks-hpc"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings("ignore")

print("Python:", sys.executable)

from dotenv import dotenv_values
llm_config = dotenv_values('/nvme/scratch/cy26nk2/llm.env')
embd_config = dotenv_values('/nvme/scratch/cy26nk2/embd.env')
print("LLM model loaded:", llm_config.get("MODEL_NAME", "<missing>"))
print("Embedder model loaded:", embd_config.get("MODEL_NAME", "<missing>"))

from results_logger import save_result_row, clear_results_for_implementation

RESULTS_CSV = PROJECT_ROOT / "data" / "results" / "experimental" / "hpc_results.csv"
# Start this implementation's results from a clean slate.
clear_results_for_implementation("vectordb_tuned", results_csv=RESULTS_CSV)

from haystack import Pipeline
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter, DocumentCleaner

from milvus_haystack import MilvusDocumentStore
from milvus_haystack.milvus_embedding_retriever import MilvusEmbeddingRetriever

from haystack.components.embedders import OpenAIDocumentEmbedder, OpenAITextEmbedder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.utils import Secret

from haystack_integrations.components.converters.docling import DoclingConverter
from docling.chunking import HybridChunker

print("Imports work")

Python: /nvme/h/cy26nk2/.conda/envs/notebookEnv/bin/python3.14
LLM model loaded: meta-llama/Llama-3.1-8B-Instruct
Embedder model loaded: BAAI/bge-m3
Cleared existing rows for implementation='vectordb_tuned'
Imports work


## Connect to Milvus

This notebook uses a file-backed Milvus URI for local persistence in the project workspace.  
This keeps indexing reproducible across reruns in HPC sessions.

In [2]:
HOME_DIR = os.path.expanduser("~")
db_path = os.path.join(HOME_DIR, "Thesis/notebooks-hpc", "rag_vectordb_tuned.db")

In [3]:
document_store = MilvusDocumentStore(
    connection_args={"uri": db_path},
    drop_old=True,
    enable_dynamic_field=False,
)

print("Milvus ready")

Milvus ready


## Load the OrchestrAI dataset

The corpus is organized by department under **`data/dummy_data/`**; all `.docx` files are loaded recursively.

In [4]:
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "dummy_data"
FILES = [str(f.resolve()) for f in DOCUMENTS_DIR.rglob("*.docx")]

print("Files found:", len(FILES))
for f in FILES[:10]:
    print("-", Path(f).name)

Files found: 54
- VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx
- VAL-SALES-005_Discount_Exception_Policy_refreshed.docx
- VAL-SALES-001_Lead_Routing_Rules_refreshed.docx
- VAL-SALES-004_CRM_Field_Governance_Note_refreshed.docx
- VAL-SALES-003_Campaign_Approval_Workflow_refreshed.docx
- VAL-LOG-005_Reverse_Logistics_Routing_Memo.docx
- VAL-LOG-004_Regional_Delivery_SLA_Matrix.docx
- VAL-LOG-001_Carrier_Delay_Escalation_Notice.docx
- VAL-LOG-002_Warehouse_Receiving_Checklist.docx
- VAL-LOG-003_Damaged_Goods_Handling_SOP.docx


## Advanced indexing strategy

To make the advanced pipeline reliable, the indexing flow is:

1. Convert each file with `DoclingConverter`
2. Attach explicit source metadata (`file_name`, `file_path`, `department`)
3. Apply an additional `DocumentSplitter` before embedding
4. Remove metadata fields that Milvus cannot store cleanly
5. Generate embeddings with vLLM-served embedder
6. Write the embedded chunks to Milvus

This file-by-file indexing approach fixes the earlier `unknown source` problem during retrieval.

In [5]:
OLLAMA_EMBED_MODEL = "mxbai-embed-large"
HF_TOKENIZER_ID = "mixedbread-ai/mxbai-embed-large-v1"

converter = DoclingConverter(
    chunker=HybridChunker(
        tokenizer=HF_TOKENIZER_ID,
        max_tokens=400,
    )
)

doc_embedder = OpenAIDocumentEmbedder(
    api_key=Secret.from_token(embd_config["VLLM_API_KEY"]),
    api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
    model=embd_config["MODEL_NAME"],
)

writer = DocumentWriter(document_store=document_store)

def normalize_meta(meta):
    cleaned = {}
    for key, value in (meta or {}).items():
        if key == "_split_overlap":
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            cleaned[key] = value
        else:
            cleaned[key] = str(value)
    return cleaned

indexing_start = time.perf_counter()

for file_path in FILES:
    file_path = Path(file_path)

    result = converter.run(sources=[file_path])
    converted_docs = result["documents"]

    for doc in converted_docs:
        if doc.meta is None:
            doc.meta = {}
        doc.meta["file_name"] = file_path.name
        doc.meta["file_path"] = str(file_path.resolve())
        doc.meta["department"] = file_path.parent.name

    cleaner = DocumentCleaner()
    cleaned_docs = cleaner.run(documents=converted_docs)["documents"]

    for doc in cleaned_docs:
        if doc.meta is None:
            doc.meta = {}
        doc.meta["file_name"] = file_path.name
        doc.meta["file_path"] = str(file_path.resolve())
        doc.meta["department"] = file_path.parent.name

    splitter = DocumentSplitter(
        split_by="word",
        split_length=200,
        split_overlap=40
    )

    split_docs = splitter.run(documents=cleaned_docs)["documents"]

    for doc in split_docs:
        if doc.meta is None:
            doc.meta = {}
        doc.meta["file_name"] = file_path.name
        doc.meta["file_path"] = str(file_path.resolve())
        doc.meta["department"] = file_path.parent.name
        doc.meta = normalize_meta(doc.meta)

    docs_with_embeddings = doc_embedder.run(split_docs)
    writer.run(documents=docs_with_embeddings["documents"])

indexing_time_s = round(time.perf_counter() - indexing_start, 4)

print("Indexing time (s):", indexing_time_s)
print(f"Indexed {len(FILES)} source files into Milvus.")

Calculating embeddings: 1it [00:00,  2.03it/s]
Calculating embeddings: 1it [00:00, 32.94it/s]
Calculating embeddings: 1it [00:00, 28.72it/s]
Calculating embeddings: 1it [00:00, 31.94it/s]
Calculating embeddings: 1it [00:00, 32.02it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (607 > 512). Running this sequence through the model will result in indexing errors
Calculating embeddings: 1it [00:00, 27.93it/s]
Calculating embeddings: 1it [00:00, 25.69it/s]
Calculating embeddings: 1it [00:00, 24.47it/s]
Calculating embeddings: 1it [00:00, 18.11it/s]
Calculating embeddings: 1it [00:00, 27.26it/s]
Calculating embeddings: 1it [00:00, 32.57it/s]
Calculating embeddings: 1it [00:00, 32.20it/s]
Calculating embeddings: 1it [00:00, 32.64it/s]
Calculating embeddings: 1it [00:00, 32.22it/s]
Calculating embeddings: 1it [00:00, 31.78it/s]
Calculating embeddings: 1it [00:00, 30.99it/s]
Calculating embeddings: 1it [00:00, 31.31it/s]
Calc

Indexing time (s): 12.8415
Indexed 54 source files into Milvus.


## Build the vector RAG pipeline

The answer-generation logic is aligned with the basic notebook:
- use only the retrieved context
- do not invent missing information
- cite source filenames when possible

In [6]:
template = [
    ChatMessage.from_user(
        """
Respond to the User Query using only the provided Context.

General Guidelines:
- Use only the provided context.
- If the answer is not found in the context, say so clearly.
- Do not make up facts, document sections, pages, or citations.
- If the answer comes from multiple documents, cite all relevant documents.
- Respond in the same language as the user’s query.
- Do not use emojis.
- Be professional and concise.
- Do not write a conclusion or follow-up unless the user asks for it.
- If the context contains a direct policy rule, exception, SLA, or escalation instruction, state it directly and do not hedge.
- Prefer the most specific procedural source over broader handbook summaries when both are present.
- If the retrieved context does not explicitly contain the rule, exception, owner, or next step needed to answer the question, say that the retrieved context is insufficient instead of inferring or guessing.
- Do not mention system names, tools, workflows, or teams unless they are explicitly present in the retrieved context.

Citation Rules:
- For every important claim, cite the source document filename.
- If available in the retrieved context, also mention the document ID or department.
- Do not mention source chapter, section, or page unless they are explicitly present in the retrieved context.

Context:
{% for document in documents %}
[Source: {{ document.meta.get("file_name", document.meta.get("file_path", "unknown source")) }}]
{{ document.content }}

{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

text_embedder = OpenAITextEmbedder(
    api_key=Secret.from_token(embd_config["VLLM_API_KEY"]),
    api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
    model=embd_config["MODEL_NAME"],
)

retriever = MilvusEmbeddingRetriever(
    document_store=document_store,
    top_k=10
)

chat_generator = OpenAIChatGenerator(
    api_key=Secret.from_token(llm_config["VLLM_API_KEY"]),
    api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
    model=llm_config["MODEL_NAME"],
    generation_kwargs={
        "temperature": 0.0,
        "top_p": 1.0,
        "seed": 42,
    }
)

prompt_builder = ChatPromptBuilder(
    template=template,
    required_variables=["question", "documents"]
)

advanced_rag_pipeline = Pipeline()
advanced_rag_pipeline.add_component("text_embedder", text_embedder)
advanced_rag_pipeline.add_component("retriever", retriever)
advanced_rag_pipeline.add_component("prompt_builder", prompt_builder)
advanced_rag_pipeline.add_component("llm", chat_generator)

advanced_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
advanced_rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
advanced_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

print("Vector RAG pipeline ready")

Vector RAG pipeline ready


## Helper function for presentation queries

The function below:
- embeds the question
- retrieves the top document chunks
- prints the retrieved sources and snippets
- runs the full vector RAG pipeline
- prints the final answer

In [7]:
def run_vectordb_query(question):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("=" * 80)

    retrieval_start = time.perf_counter()

    q_emb = text_embedder.run(text=question)["embedding"]
    retr_out = retriever.run(query_embedding=q_emb, top_k=10)
    top_docs = retr_out["documents"]

    retrieval_time_s = round(time.perf_counter() - retrieval_start, 4)

    print(f"\nRETRIEVED DOCUMENTS: {len(top_docs)}")
    print(f"RETRIEVAL TIME (s): {retrieval_time_s}\n")

    for i, d in enumerate(top_docs, 1):
        source = d.meta.get("file_name", d.meta.get("file_path", "unknown source"))
        department = d.meta.get("department", "unknown department")
        snippet = (d.content or "")[:400].replace("\n", " ")
        print(f"[{i}] Source: {source} | Department: {department}")
        print(f"    {snippet}...")
        print()

    generation_start = time.perf_counter()

    result = advanced_rag_pipeline.run({
        "text_embedder": {"text": question},
        "retriever": {"top_k": 10},
        "prompt_builder": {"question": question},
    })

    answer = result["llm"]["replies"][0].text
    generation_time_s = round(time.perf_counter() - generation_start, 4)

    print("=" * 80)
    print("FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print(f"\nGENERATION TIME (s): {generation_time_s}")

    return top_docs, answer, retrieval_time_s, generation_time_s

# Saving results in CSV

In [8]:
def run_and_save_vectordb(question, scenario_id):
    top_docs, answer, retrieval_time_s, generation_time_s = run_vectordb_query(question)

    save_result_row(
        implementation="vectordb_tuned",
        scenario_id=scenario_id,
        question=question,
        generated_answer=answer,
        top_docs=top_docs,
        retriever_top_k=10,
        reranker_top_k=None,
        indexing_time_s=indexing_time_s,
        retrieval_time_s=retrieval_time_s,
        generation_time_s=generation_time_s,
        results_csv=RESULTS_CSV,
    )

    return top_docs, answer

### Clear vectordb rows

In [9]:
clear_results_for_implementation("vectordb_tuned", results_csv=RESULTS_CSV)

Cleared existing rows for implementation='vectordb_tuned'


PosixPath('/nvme/h/cy26nk2/Thesis/data/results/experimental/hpc_results.csv')

### Scenario 1 — Basic fact retrieval
This is the clean baseline. It shows whether the system can retrieve one exact operational fact correctly.
- Single-document, single-fact retrieval.
- Tests basic embedding-based retrieval accuracy.
- Expected answer should come from the Customer Success / Support materials.

In [10]:
top_docs_vectordb_s1, answer_vectordb_s1 = run_and_save_vectordb(
    'What is the first-response SLA for a Sev-1 support incident?',
    scenario_id="vt1",
)

QUESTION:
What is the first-response SLA for a Sev-1 support incident?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1699

[1] Source: VAL-CS-001_Severity_Matrix.docx | Department: Customer Support
    Severity Matrix Classification rules Sev-1, **Typical condition** = Critical production outage at one or more active sites. Sev-1, **Examples** = Pick-path execution blocked, gateway fleet offline, order waves cannot be released. Sev-1, **Response target** = 15 minutes. Sev-1, **Escalate to** = Support manager, Engineering incident lead, CSM. Sev-2, **Typical condition** = Major degradation with n...

[2] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    Customer Success and Support Handbook Service targets and escalation guide Sev-1 first response, **Target** = 15 min. Sev-1 first response, **Reviewed by** = Support manager. Sev-1 first response, **Notes** = Clock starts when the case is correctly classified. Sev-2 first response

### Scenario 2 — Procedural retrieval
This scenario is more complex because the answer is not a single fact, but a short operational process.
- Procedure and checklist retrieval.
- Tests whether the system can retrieve and summarize structured steps clearly.
- Expected answer should come mainly from the HR / People Operations materials.

In [11]:
top_docs_vectordb_s2, answer_vectordb_s2 = run_and_save_vectordb(
    'What steps must be completed before a new employee is considered ready for day one at OrchestrAI?',
    scenario_id="vt2",
)

QUESTION:
What steps must be completed before a new employee is considered ready for day one at OrchestrAI?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1641

[1] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
    Day-One Onboarding Checklist **Document ID:** VAL-HR-001 **Owner:** HR & People Operations **Confidentiality:** Internal Use Only Purpose: This checklist standardizes the minimum tasks that must be complete before an OrchestrAI employee can be considered ready for day-one onboarding. **Department**, **Value** = HR & People Operations. **Effective date**, **Value** = 2026-02-24. **Applies to**, **V...

[2] Source: VAL-PROD-001_Launch_Readiness_Checklist.docx | Department: Product
    **OrchestrAI Team Knowledge Base** **Launch Readiness Checklist** Operational checklist used by Product and PMO to confirm whether a customer-facing release may proceed to go-live. **Effective Date** 2026-03-24, **Department** Product & Program Management = **Review Cycle*

### Scenario 3 — Multi-team operational reasoning
This scenario requires combining information from more than one team to answer a realistic workplace problem.
- Multi-document, cross-team retrieval.
- Tests whether the system can synthesize HR, IT, and Security responsibilities.
- Expected answer should identify both the involved teams and the next actions.

In [12]:
top_docs_vectordb_s3, answer_vectordb_s3 = run_and_save_vectordb(
    'A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?',
    scenario_id="vt3",
)

QUESTION:
A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1638

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    Common Questions **Q: How far in advance should a laptop be requested?** A: Standard new-hire laptop requests should be present in the hiring workflow at least seven business days before the employee start date. International shipments, executive hires, and custom software bundles may require ten to fifteen business days. **Q: Which team starts the request?** A: The request begins with the hiring ...

[2] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
    Decision and Escalation Reference Start date within 72 hours and no assigned device, **Primary Owner** = IT Service Desk Lead. Start date within 72 hours and no assigned device, **Escalation Trigger** = Notify hiring manag

### Scenario 4 — Policy + exception handling
This scenario checks whether the system can distinguish the normal rule from the emergency exception path.
- Policy retrieval with exception reasoning.
- Tests whether the system can identify when a standard workflow may be bypassed.
- Expected answer should combine Finance / Procurement rules with operational urgency.

In [13]:
top_docs_vectordb_s4, answer_vectordb_s4 = run_and_save_vectordb(
    'Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?',
    scenario_id="vt4",
)

QUESTION:
Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1634

[1] Source: TEAMQA-FIN-001_OrchestrAI_Finance_and_Procurement_Handbook.docx | Department: Q&A Hanbooks
    Finance and Procurement Handbook Specialised Q&A **Q: What does Finance and Procurement own at OrchestrAI?** **A:** The team owns vendor onboarding, purchase-order governance, invoice and payment controls, employee reimbursement policy, budget coordination, and month-end readiness. It is also the main gatekeeper for spend discipline and exception documentation. **Q: When is a purchase order requir...

[2] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    OrchestrAI Logistics and Field Fulfillment Handbook 2. Team Mission and Operating Context The Logistics and Field Fulfillment team ensures that OrchestrAI hardware and deployment kits mov

### Scenario 5 — Cross-functional enterprise reasoning
This is the most complex scenario because it requires cross-functional reasoning across several business units.
- Multi-hop, multi-document retrieval.
- Tests whether the system can combine support, product, logistics, and commercial risk signals.
- Expected answer should identify the teams involved and propose a coordinated next-step plan.

In [14]:
top_docs_vectordb_s5, answer_vectordb_s5 = run_and_save_vectordb(
    'A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?',
    scenario_id="vt5",
)

QUESTION:
A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.1646

[1] Source: TEAMQA-FIN-001_OrchestrAI_Finance_and_Procurement_Handbook.docx | Department: Q&A Hanbooks
    Security review a vendor?** **A:** Security review is required when the vendor will access OrchestrAI systems, process internal data, host critical workflows, or support services that create operational or privacy risk. **Q: How are hardware purchases handled differently from software subscriptions?** **A:** Hardware purchases often require receiving confirmation, serial tracking, shipping coordin...

[2] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    OrchestrAI Logistics and Field Fulfillment Handbook 5. Operational Questions and Answers What data